# Wireline launch analysis — 2026-08-13

## TL;DR

Wireline's production architecture and data controls are strong, but the audited account base contains no production workspaces or Stripe-sourced subscription. The pre-launch catalog also allowed independent full use of voice, Runtime Allowance, and managed AI to breach the margin floor. The launch-safe catalog correction restores at least 30% full-use contribution on every monthly and annual plan without changing prices.

## Context and methods

This notebook uses bounded aggregate evidence from the live Supabase and Stripe audits plus constants from `src/lib/plan-catalog.ts`. It intentionally excludes credentials, user-level records, emails, phone numbers, and provider payloads. Values are a point-in-time snapshot, not a live connector.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

snapshot_at = '2026-08-13'
account_environment = pd.DataFrame([
    {'environment': 'Test', 'accounts': 134},
    {'environment': 'Demo', 'accounts': 1},
    {'environment': 'Production', 'accounts': 0},
])
controls = {
    'public_tables': 82,
    'public_tables_with_rls': 82,
    'active_stripe_sourced_subscriptions': 0,
    'unprocessed_or_abandoned_stripe_events': 0,
    'duplicate_stripe_subscription_ids': 0,
    'subscriptions_missing_entitlement_source': 0,
    'overdue_deletion_queue_items': 0,
}
account_environment, controls

## Data validation

The checks below prove the snapshot is internally consistent and that no production-adoption claim can be inferred from test/demo activity.

In [ ]:
assert account_environment['accounts'].sum() == 135
assert int(account_environment.loc[account_environment.environment == 'Production', 'accounts'].iloc[0]) == 0
assert controls['public_tables'] == controls['public_tables_with_rls']
assert controls['active_stripe_sourced_subscriptions'] == 0
assert controls['unprocessed_or_abandoned_stripe_events'] == 0
print('Snapshot validation passed.')

## Results: reachable plan economics

The old model reused the voice reserve to size a separately metered managed-AI budget. Because the three pools could all be consumed in one month, the reachable annual contribution margin was negative. The corrected model explicitly reserves all pools and reduces included voice capacity to the level the existing price can safely fund.

In [ ]:
VOICE_COST_PER_MIN = 0.128
PHONE_NUMBER_COST = 1.15
PAYMENT_RATE = 0.029
BILLING_RATE = 0.007
TAX_RESERVE_RATE = 0.005
PAYMENT_FIXED = 0.30
HARD_MARGIN_FLOOR = 0.30

plans = pd.DataFrame([
    {'plan': 'Starter', 'monthly': 19, 'annual': 190, 'numbers': 1, 'runtime': 1, 'old_minutes': 60, 'old_ai': 8.68, 'new_minutes': 50, 'new_ai': 1.50},
    {'plan': 'Starter Plus', 'monthly': 39, 'annual': 390, 'numbers': 2, 'runtime': 3, 'old_minutes': 120, 'old_ai': 18.36, 'new_minutes': 100, 'new_ai': 3.00},
    {'plan': 'Pro', 'monthly': 79, 'annual': 790, 'numbers': 5, 'runtime': 6, 'old_minutes': 245, 'old_ai': 37.36, 'new_minutes': 190, 'new_ai': 6.00},
    {'plan': 'Business', 'monthly': 159, 'annual': 1590, 'numbers': 15, 'runtime': 12, 'old_minutes': 450, 'old_ai': 69.60, 'new_minutes': 350, 'new_ai': 12.00},
])

def stripe_fees(gross):
    return gross * (PAYMENT_RATE + BILLING_RATE + TAX_RESERVE_RATE) + PAYMENT_FIXED

def full_margin(row, interval, version):
    revenue = row['monthly'] if interval == 'month' else row['annual'] / 12
    fees = stripe_fees(row['monthly']) if interval == 'month' else stripe_fees(row['annual']) / 12
    cost = row[f'{version}_minutes'] * VOICE_COST_PER_MIN + row['numbers'] * PHONE_NUMBER_COST + row['runtime'] + row[f'{version}_ai'] + fees
    return (revenue - cost) / revenue

for version in ['old', 'new']:
    for interval in ['month', 'year']:
        plans[f'{version}_{interval}_margin'] = plans.apply(lambda row: full_margin(row, interval, version), axis=1)

assert (plans['new_month_margin'] >= HARD_MARGIN_FLOOR).all()
assert (plans['new_year_margin'] >= HARD_MARGIN_FLOOR).all()
assert (plans['old_year_margin'] < 0).all()
margin_summary = plans[['plan', 'old_year_margin', 'new_year_margin']].copy()
margin_summary['old_year_margin'] = margin_summary['old_year_margin'].map(lambda value: f'{value:.1%}')
margin_summary['new_year_margin'] = margin_summary['new_year_margin'].map(lambda value: f'{value:.1%}')
print(margin_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

env_plot = account_environment.sort_values('accounts')
axes[0].barh(env_plot['environment'], env_plot['accounts'], color='#8B5CF6')
axes[0].set_title('Workspace environment mix')
axes[0].set_xlabel('Accounts in audited snapshot')
axes[0].spines[['top', 'right']].set_visible(False)

x = range(len(plans))
width = 0.36
axes[1].bar([i - width / 2 for i in x], plans['old_year_margin'] * 100, width, label='Before correction', color='#B6B1C6')
axes[1].bar([i + width / 2 for i in x], plans['new_year_margin'] * 100, width, label='Launch-safe', color='#8B5CF6')
axes[1].axhline(30, color='#171426', linewidth=1.2, linestyle='--', label='30% floor')
axes[1].axhline(0, color='#8B87A0', linewidth=0.8)
axes[1].set_title('Annual full-use contribution margin')
axes[1].set_ylabel('Margin (%)')
axes[1].set_xticks(list(x), plans['plan'], rotation=20, ha='right')
axes[1].legend(frameon=False, fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)
plt.show()

## Takeaways

1. **Technical posture:** 82 of 82 public tables have RLS; the audited billing, entitlement, deletion, and idempotency integrity checks are clean.
2. **Commercial posture:** there are zero production workspaces and zero Stripe-sourced active subscriptions, so the product is commercially unvalidated despite substantial test/demo data.
3. **Economics:** the launch-safe catalog turns a reachable negative-margin state into 30.9%–32.3% annual full-use contribution across paid plans.
4. **Decision:** use a controlled, consent-led design-partner launch until the first real paid lifecycle and production activation are proven; do not interpret test/demo records as demand.